# Hyperparameter Tuning and Optimization Pipeline
### Internship Track: Week 3 – Task 1
**Author:** Sayantani Adhikary  
**Objective:** Optimize a Random Forest Classifier using GridSearchCV and RandomizedSearchCV, evaluate hyperparameter impacts on accuracy, and analyze computational efficiency trade-offs.


## 1. Import Libraries & Load Data
We import Pandas, NumPy, Scikit-Learn tuning modules, and plotting libraries.


In [1]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

sns.set_theme(style="whitegrid")
df = pd.read_csv('../data/dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()


## 2. Preprocessing & Holdout Split
Separate features ($X$) and target ($y$), apply `StandardScaler`, and reserve 20% of data for final testing.


In [1]:
X = df.drop(columns=['DefaultRisk'])
y = df['DefaultRisk']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")


## 3. Baseline Model (Untuned Random Forest)
Fit an untuned baseline Random Forest using default parameters to establish a reference benchmark.


In [1]:
base_rf = RandomForestClassifier(random_state=42)
base_rf.fit(X_train, y_train)
base_preds = base_rf.predict(X_test)

base_acc = accuracy_score(y_test, base_preds)
base_f1 = f1_score(y_test, base_preds, average='weighted')
print(f"Baseline Test Accuracy: {base_acc * 100:.2f}%")
print(f"Baseline F1-Score:      {base_f1:.4f}")


## 4. Hyperparameter Optimization: RandomizedSearchCV vs. GridSearchCV
Define the hyperparameter search space across `n_estimators`, `max_depth`, `min_samples_split`, and `min_samples_leaf`.


In [1]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# 1. Randomized Search
start_rand = time.time()
rand_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=8,
    cv=cv,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
rand_search.fit(X_train, y_train)
rand_time = time.time() - start_rand

# 2. Grid Search
start_grid = time.time()
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - start_grid

print(f"RandomizedSearchCV Time: {rand_time:.2f}s | Best Params: {rand_search.best_params_}")
print(f"GridSearchCV Time:       {grid_time:.2f}s | Best Params: {grid_search.best_params_}")


## 5. Performance Comparison & Evaluation
Compare accuracy, F1-scores, and training efficiency across configurations.


In [1]:
models = {
    'Baseline RF': base_rf,
    'RandomizedSearch RF': rand_search.best_estimator_,
    'GridSearch RF (Optimal)': grid_search.best_estimator_
}

comparison_data = []
for name, m in models.items():
    preds = m.predict(X_test)
    comparison_data.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, preds, average='weighted'),
        'F1-Score': f1_score(y_test, preds, average='weighted')
    })

scorecard = pd.DataFrame(comparison_data)
print("=== Performance Comparison Scorecard ===")
print(scorecard.to_string(index=False))


## 6. Visualization: Search Runtime vs Accuracy Trade-off
Visualize the efficiency trade-off between exhaustive Grid Search and stochastic Randomized Search.


In [1]:
plt.figure(figsize=(8, 4.5))
methods = ['RandomizedSearchCV', 'GridSearchCV']
runtimes = [rand_time, grid_time]

sns.barplot(x=methods, y=runtimes, palette='viridis')
plt.title('Hyperparameter Search Execution Time Comparison', fontsize=12, fontweight='bold')
plt.ylabel('Runtime (Seconds)')
for i, t in enumerate(runtimes):
    plt.text(i, t + 0.05, f"{t:.2f}s", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
